# Сравнение FP32 и AMP FP16

Запустите все ячейки сверху вниз. Два независимых обучения выполняются
последовательно. Начальные веса, данные, размер батча и число эпох одинаковые.
Валидация в обоих режимах проходит в FP32. Для пробного запуска можно снизить
`EPOCHS` в ячейке настроек. Для воспроизведения итогового сравнения используйте 130.

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd()
if not (ROOT / "utils" / "alexnet.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "utils" / "alexnet.py").is_file():
    raise RuntimeError("Откройте ноутбук из корня проекта или из папки notebooks.")
sys.path.insert(0, str(ROOT))

from utils.data_utils import load_training_data

DATA_DIR = ROOT / "data"
ARTIFACTS = ROOT / "artifacts"
SEED = 52
BATCH_SIZE = 512

if not torch.cuda.is_available():
    raise RuntimeError("Для этих экспериментов нужна CUDA. Проверьте выбранное ядро.")
DEVICE = torch.device("cuda")
print(torch.cuda.get_device_name(DEVICE), "PyTorch", torch.__version__)

from utils.train_utils import fit

EPOCHS = 130
WORKERS = 2
data = load_training_data(DATA_DIR, SEED)

In [ ]:
fp32_history = fit(
    data, DEVICE, ARTIFACTS / "fp32",
    epochs=EPOCHS, seed=SEED, batch_size=BATCH_SIZE, workers=WORKERS,
)

In [ ]:
amp_history = fit(
    data, DEVICE, ARTIFACTS / "amp_fp16",
    epochs=EPOCHS, seed=SEED, batch_size=BATCH_SIZE, workers=WORKERS, use_amp=True,
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def summarize(history):
    best = history.loc[history["val_acc"].idxmax()]
    return {
        "best_val_acc_pct": 100 * best["val_acc"],
        "best_epoch": int(best["epoch"]),
        "last_val_acc_pct": 100 * history["val_acc"].iloc[-1],
        "last_5_val_acc_pct": 100 * history["val_acc"].tail(5).mean(),
        "train_val_seconds": history["seconds"].sum(),
    }


summary = pd.DataFrame({
    "FP32": summarize(fp32_history),
    "AMP FP16": summarize(amp_history),
}).T
summary.to_csv(ARTIFACTS / "comparison.csv")
display(summary.round(2))

delta = summary.loc["AMP FP16", "last_5_val_acc_pct"] - summary.loc["FP32", "last_5_val_acc_pct"]
speedup = summary.loc["FP32", "train_val_seconds"] / summary.loc["AMP FP16", "train_val_seconds"]
print(f"AMP − FP32 за последние пять эпох {delta:+.2f} п.п.")
print(f"Ускорение train + val {speedup:.2f}×")

fig, (by_epoch, by_time) = plt.subplots(1, 2, figsize=(12, 4))
for label, history in (("FP32", fp32_history), ("AMP FP16", amp_history)):
    by_epoch.plot(history["epoch"], history["val_acc"], label=label)
    by_time.plot(history["seconds"].cumsum() / 60, history["val_acc"], label=label)

by_epoch.set(xlabel="Эпоха", ylabel="Точность на validation")
by_time.set(xlabel="Время train + val, мин", ylabel="Точность на validation")
for ax in (by_epoch, by_time):
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(ARTIFACTS / "validation_accuracy.png", dpi=160)
plt.show()